# Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import precision_score
from sklearn.model_selection import cross_validate
from sklearn.model_selection import cross_val_predict

from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import BaggingClassifier
from sklearn.svm import SVC
from sklearn.ensemble import StackingClassifier

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv


# Dataset Loading

In [ ]:
train = pd.read_csv(r'D:\Project Phase II\Dataset\finaltrain.csv',encoding='cp1252')
test = pd.read_csv(r'D:\Project Phase II\Dataset\finaltest.csv',encoding='cp1252') #test_set1 
autrain = pd.read_csv(r'D:\Project Phase II\Dataset\UNSW_NB15_training-set.csv',encoding='cp1252') #train_set2
autest = pd.read_csv(r'D:\Project Phase II\Dataset\UNSW_NB15_testing-set.csv',encoding='cp1252') #test_set2

# Input Transformation

## Training

In [ ]:
nodes = set()
trainUnique = train
for i in range(len(train)):
    src = str(train['srcip'].iloc[i])+':'+ str(train['sport'].iloc[i]) + ':' + str(train['sttl'].iloc[i])+':'+str(train['swin'].iloc[i])
    dst = str(train['dstip'].iloc[i])+':'+str(train['dsport'].iloc[i]) + ':' + str(train['dttl'].iloc[i])+':'+str(train['dwin'].iloc[i])
    nodes.add(src)
    nodes.add(dst)


#Find unique (srcip,sport,dstip,dsport) tuples in train
train_tuples = set()
for i in range(len(train)):
    train_tuples.add((train['srcip'].iloc[i], train['sport'].iloc[i], train['dstip'].iloc[i], train['dsport'].iloc[i], train['sttl'].iloc[i], train['swin'].iloc[i], train['dttl'].iloc[i], train['dwin'].iloc[i]))

In [ ]:
xLookUp = {}
x = []
count = 0
edge_index = []
edge_attr = []
for i in range(len(train)):
    src = str(train['srcip'].iloc[i])+':'+ str(train['sport'].iloc[i]) + ':' + str(train['sttl'].iloc[i])+':'+str(train['swin'].iloc[i])
    dst = str(train['dstip'].iloc[i])+':'+str(train['dsport'].iloc[i]) + ':' + str(train['dttl'].iloc[i])+':'+str(train['dwin'].iloc[i])
    if src not in xLookUp:
        xLookUp[src] = count
        # x.append([int(train['sport'].iloc[i]), int(train['sttl'].iloc[i]), int(train['swin'].iloc[i])])
        x.append([int(train['sttl'].iloc[i]), int(train['swin'].iloc[i])])
        count += 1
    if dst not in xLookUp:
        xLookUp[dst] = count
        # x.append([int(train['dsport'].iloc[i]), int(train['dttl'].iloc[i]), int(train['dwin'].iloc[i])])
        x.append([int(train['dttl'].iloc[i]), int(train['dwin'].iloc[i])])
        count += 1
    edge_index.append([xLookUp[src], xLookUp[dst]])
    edge_attr.append(list(train.iloc[i].drop(['srcip', 'sport', 'dstip', 'dsport', 'sttl', 'dttl','swin','dwin','Label']).values))


# Convert x to tensor
x_tensor = torch.tensor(x, dtype=torch.float)

# Convert edge_index to tensor
edge_index_tensor = torch.tensor(edge_index, dtype=torch.long).t()  # Transpose for correct format

# Convert edge_attr to tensor
edge_attr_tensor = torch.tensor(edge_attr, dtype=torch.float)

print(x_tensor.shape, edge_index_tensor.shape, edge_attr_tensor.shape)

## Testing

In [ ]:
testnodes = set()
testUnique = test
for i in range(len(test)):
    src = str(test['srcip'].iloc[i])+':'+ str(test['sport'].iloc[i]) + ':' + str(test['sttl'].iloc[i])+':'+str(test['swin'].iloc[i])
    dst = str(test['dstip'].iloc[i])+':'+str(test['dsport'].iloc[i]) + ':' + str(test['dttl'].iloc[i])+':'+str(test['dwin'].iloc[i])
    testnodes.add(src)
    testnodes.add(dst)


#Find unique (srcip,sport,dstip,dsport) tuples in train
test_tuples = set()
for i in range(len(test)):
    test_tuples.add((test['srcip'].iloc[i], test['sport'].iloc[i], test['dstip'].iloc[i], test['dsport'].iloc[i], test['sttl'].iloc[i], test['swin'].iloc[i], test['dttl'].iloc[i], test['dwin'].iloc[i]))

In [ ]:
xtestLookUp = {}
xtest = []
count = 0
edge_index_test = []
edge_attr_test = []
for i in range(len(test)):
    src = str(test['srcip'].iloc[i])+':'+ str(test['sport'].iloc[i]) + ':' + str(test['sttl'].iloc[i])+':'+str(test['swin'].iloc[i])
    dst = str(test['dstip'].iloc[i])+':'+str(test['dsport'].iloc[i]) + ':' + str(test['dttl'].iloc[i])+':'+str(test['dwin'].iloc[i])
    if src not in xtestLookUp:
        xtestLookUp[src] = count
        # xtest.append([int(test['sport'].iloc[i]), int(test['sttl'].iloc[i]), int(test['swin'].iloc[i])])
        xtest.append([int(test['sttl'].iloc[i]), int(test['swin'].iloc[i])])
        count += 1
    if dst not in xtestLookUp:
        xtestLookUp[dst] = count
        # xtest.append([int(test['dsport'].iloc[i]), int(test['dttl'].iloc[i]), int(test['dwin'].iloc[i])])
        xtest.append([int(test['dttl'].iloc[i]), int(test['dwin'].iloc[i])])
        count += 1
    edge_index_test.append([xtestLookUp[src], xtestLookUp[dst]])
    edge_attr_test.append(list(test.iloc[i].drop(['srcip', 'sport', 'dstip', 'dsport', 'sttl', 'dttl','swin','dwin','Label']).values))


# Convert x to tensor
x_test_tensor = torch.tensor(xtest, dtype=torch.float)

# Convert edge_index to tensor
edge_index_test_tensor = torch.tensor(edge_index_test, dtype=torch.long).t()  # Transpose for correct format

# Convert edge_attr to tensor
edge_attr_test_tensor = torch.tensor(edge_attr_test, dtype=torch.float)

print(x_test_tensor.shape, edge_index_test_tensor.shape, edge_attr_test_tensor.shape)

## Author's Training Dataset

Since srcip, sport, dstip, dsport information is not available, we proceed with having each node feature value as 0, and assume each edge as an unique connection between two imaginary nodes, we add stcpb and dtcpb as placeholders 

In [ ]:
autrainnodes = set()
autrainUnique = autrain
for i in range(len(autrain)):
    src = str(autrain['sttl'].iloc[i])+':'+str(autrain['swin'].iloc[i]) + str(autrain['stcpb'].iloc[i]) + str(autrain['sbytes'].iloc[i]) + str(autrain['Sload'].iloc[i])+ str(autrain['Spkts'].iloc[i])+ str(autrain['smeansz'].iloc[i])+ str(autrain['Sjit'].iloc[i]) + str(autrain['Sintpkt'].iloc[i])
    dst = str(autrain['dttl'].iloc[i])+':'+str(autrain['dwin'].iloc[i]) + str(autrain['dtcpb'].iloc[i]) + str(autrain['dbytes'].iloc[i]) + str(autrain['Dload'].iloc[i])+ str(autrain['Dpkts'].iloc[i])+ str(autrain['dmeansz'].iloc[i])+ str(autrain['Djit'].iloc[i]) + str(autrain['Dintpkt'].iloc[i])
    autrainnodes.add(src)
    autrainnodes.add(dst)


#Find unique (srcip,sport,dstip,dsport) tuples in train
autrain_tuples = set()
for i in range(len(autrain)):
    autrain_tuples.add((autrain['sttl'].iloc[i], autrain['swin'].iloc[i], autrain['dttl'].iloc[i], autrain['dwin'].iloc[i], autrain['stcpb'].iloc[i], autrain['dtcpb'].iloc[i], autrain['sbytes'].iloc[i], autrain['dbytes'].iloc[i], autrain['Sload'].iloc[i], autrain['Dload'].iloc[i], autrain['Spkts'].iloc[i], autrain['Dpkts'].iloc[i], autrain['smeansz'].iloc[i], autrain['dmeansz'].iloc[i], autrain['Sjit'].iloc[i], autrain['Djit'].iloc[i], autrain['Sintpkt'].iloc[i], autrain['Dintpkt'].iloc[i]))

In [ ]:
xautrainLookUp = {}
xautrain = []
count = 0
edge_index_autrain = []
edge_attr_autrain = []

for i in range(len(autrain)):
    src = str(autrain['sttl'].iloc[i])+':'+str(autrain['swin'].iloc[i]) + str(autrain['stcpb'].iloc[i]) + str(autrain['sbytes'].iloc[i]) + str(autrain['Sload'].iloc[i])+ str(autrain['Spkts'].iloc[i])+ str(autrain['smeansz'].iloc[i])+ str(autrain['Sjit'].iloc[i]) + str(autrain['Sintpkt'].iloc[i])
    dst = str(autrain['dttl'].iloc[i])+':'+str(autrain['dwin'].iloc[i]) + str(autrain['dtcpb'].iloc[i]) + str(autrain['dbytes'].iloc[i]) + str(autrain['Dload'].iloc[i])+ str(autrain['Dpkts'].iloc[i])+ str(autrain['dmeansz'].iloc[i])+ str(autrain['Djit'].iloc[i]) + str(autrain['Dintpkt'].iloc[i])
    if src not in xautrainLookUp:
        xautrainLookUp[src] = count
        # xautrain.append([0,int(autrain['sttl'].iloc[i]), int(autrain['swin'].iloc[i])])
        xautrain.append([int(autrain['sttl'].iloc[i]), int(autrain['swin'].iloc[i]), int(autrain['sbytes'].iloc[i]), int(autrain['Sload'].iloc[i]), int(autrain['Spkts'].iloc[i]), int(autrain['smeansz'].iloc[i]), int(autrain['Sjit'].iloc[i]), int(autrain['Sintpkt'].iloc[i])])
        count += 1
    if dst not in xautrainLookUp:
        xautrainLookUp[dst] = count
        # xautrain.append([0, int(autrain['dttl'].iloc[i]), int(autrain['dwin'].iloc[i])])
        xautrain.append([int(autrain['dttl'].iloc[i]), int(autrain['dwin'].iloc[i]), int(autrain['dbytes'].iloc[i]), int(autrain['Dload'].iloc[i]), int(autrain['Dpkts'].iloc[i]), int(autrain['dmeansz'].iloc[i]), int(autrain['Djit'].iloc[i]), int(autrain['Dintpkt'].iloc[i])])
        count += 1
    edge_index_autrain.append([xautrainLookUp[src], xautrainLookUp[dst]])
    edge_attr_autrain.append(list(autrain.iloc[i].drop(['sttl', 'dttl','swin','dwin','Label','stcpb','dtcpb', 'sbytes', 'dbytes', 'Sload', 'Dload', 'Spkts', 'Dpkts', 'smeansz', 'dmeansz', 'Sjit', 'Djit', 'Sintpkt', 'Dintpkt']).values))



# Convert x to tensor
x_autrain_tensor = torch.tensor(xautrain, dtype=torch.float)

# Convert edge_index to tensor
edge_index_autrain_tensor = torch.tensor(edge_index_autrain, dtype=torch.long).t()  # Transpose for correct format

# Convert edge_attr to tensor
edge_attr_autrain_tensor = torch.tensor(edge_attr_autrain, dtype=torch.float)

print(x_autrain_tensor.shape, edge_index_autrain_tensor.shape, edge_attr_autrain_tensor.shape)

## Author's Testing Dataset

In [ ]:
autestnodes = set()
autestUnique = autest
for i in range(len(autest)):
    src = str(autest['sttl'].iloc[i])+':'+str(autest['swin'].iloc[i]) + str(autest['stcpb'].iloc[i]) + str(autest['sbytes'].iloc[i]) + str(autest['Sload'].iloc[i])+ str(autest['Spkts'].iloc[i])+ str(autest['smeansz'].iloc[i])+ str(autest['Sjit'].iloc[i]) + str(autest['Sintpkt'].iloc[i])
    dst = str(autest['dttl'].iloc[i])+':'+str(autest['dwin'].iloc[i]) + str(autest['dtcpb'].iloc[i]) + str(autest['dbytes'].iloc[i]) + str(autest['Dload'].iloc[i])+ str(autest['Dpkts'].iloc[i])+ str(autest['dmeansz'].iloc[i])+ str(autest['Djit'].iloc[i]) + str(autest['Dintpkt'].iloc[i])
    autestnodes.add(src)
    autestnodes.add(dst)


#Find unique (srcip,sport,dstip,dsport) tuples in train
autest_tuples = set()
for i in range(len(autest)):
    autest_tuples.add((autest['sttl'].iloc[i], autest['swin'].iloc[i], autest['dttl'].iloc[i], autest['dwin'].iloc[i], autest['stcpb'].iloc[i], autest['dtcpb'].iloc[i], autest['sbytes'].iloc[i], autest['dbytes'].iloc[i], autest['Sload'].iloc[i], autest['Dload'].iloc[i], autest['Spkts'].iloc[i], autest['Dpkts'].iloc[i], autest['smeansz'].iloc[i], autest['dmeansz'].iloc[i], autest['Sjit'].iloc[i], autest['Djit'].iloc[i], autest['Sintpkt'].iloc[i], autest['Dintpkt'].iloc[i]))

In [ ]:
xautestLookUp = {}
xautest = []
count = 0
edge_index_autest = []
edge_attr_autest = []

for i in range(len(autest)):
    src = str(autest['sttl'].iloc[i])+':'+str(autest['swin'].iloc[i]) + str(autest['stcpb'].iloc[i]) + str(autest['sbytes'].iloc[i]) + str(autest['Sload'].iloc[i])+ str(autest['Spkts'].iloc[i])+ str(autest['smeansz'].iloc[i])+ str(autest['Sjit'].iloc[i]) + str(autest['Sintpkt'].iloc[i])
    dst = str(autest['dttl'].iloc[i])+':'+str(autest['dwin'].iloc[i]) + str(autest['dtcpb'].iloc[i]) + str(autest['dbytes'].iloc[i]) + str(autest['Dload'].iloc[i])+ str(autest['Dpkts'].iloc[i])+ str(autest['dmeansz'].iloc[i])+ str(autest['Djit'].iloc[i]) + str(autest['Dintpkt'].iloc[i])
    if src not in xautestLookUp:
        xautestLookUp[src] = count
        # xautest.append([0,int(autest['sttl'].iloc[i]), int(autest['swin'].iloc[i])])
        xautest.append([int(autest['sttl'].iloc[i]), int(autest['swin'].iloc[i]), int(autest['sbytes'].iloc[i]), int(autest['Sload'].iloc[i]), int(autest['Spkts'].iloc[i]), int(autest['smeansz'].iloc[i]), int(autest['Sjit'].iloc[i]), int(autest['Sintpkt'].iloc[i])])
        count += 1
    if dst not in xautestLookUp:
        xautestLookUp[dst] = count
        # xautest.append([0, int(autest['dttl'].iloc[i]), int(autest['dwin'].iloc[i])])
        xautest.append([int(autest['dttl'].iloc[i]), int(autest['dwin'].iloc[i]), int(autest['dbytes'].iloc[i]), int(autest['Dload'].iloc[i]), int(autest['Dpkts'].iloc[i]), int(autest['dmeansz'].iloc[i]), int(autest['Djit'].iloc[i]), int(autest['Dintpkt'].iloc[i])])
        count += 1
    edge_index_autest.append([xautestLookUp[src], xautestLookUp[dst]])
    edge_attr_autest.append(list(autest.iloc[i].drop(['sttl', 'dttl','swin','dwin','Label','stcpb','dtcpb', 'sbytes', 'dbytes', 'Sload', 'Dload', 'Spkts', 'Dpkts', 'smeansz', 'dmeansz', 'Sjit', 'Djit', 'Sintpkt', 'Dintpkt']).values))



# Convert x to tensor
x_autest_tensor = torch.tensor(xautest, dtype=torch.float)

# Convert edge_index to tensor
edge_index_autest_tensor = torch.tensor(edge_index_autest, dtype=torch.long).t()  # Transpose for correct format

# Convert edge_attr to tensor
edge_attr_autest_tensor = torch.tensor(edge_attr_autest, dtype=torch.float)

print(x_autest_tensor.shape, edge_index_autest_tensor.shape, edge_attr_autest_tensor.shape)

# GAT Model

## GAT Function Declaration

nn.Linear appplies linear transformation to the incoming data: y = x * transpose(A) + b 


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv

class GATClassifier(torch.nn.Module):
    def __init__(self, num_node_features, num_edge_features, hidden_channels, num_heads, num_classes):
        super(GATClassifier, self).__init__()
        self.conv1 = GATConv(num_node_features, hidden_channels, heads=num_heads, add_self_loops=False)
        self.conv2 = GATConv(hidden_channels * num_heads, hidden_channels, heads=num_heads, add_self_loops=False)
        # self.edge_classifier = nn.Linear(hidden_channels * 2 * num_heads + num_edge_features, num_classes)
        self.edge_classifier = nn.Sequential(
        nn.Linear(hidden_channels * 2 * num_heads + num_edge_features, hidden_channels),  # Linear layer
        nn.ReLU(),  # ReLU activation function
        nn.Linear(hidden_channels, num_classes)  # Linear layer
        )

    def forward(self, x, edge_index, edge_attr):
        # Apply graph attentional layers
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        x = F.relu(x)

        # Concatenate node features with edge features
        num_edges = edge_index.size(1)
        node_x = x[edge_index[0]]
        node_x_other = x[edge_index[1]]
        edge_attr_padded = torch.cat([edge_attr, torch.zeros(num_edges - edge_attr.size(0), edge_attr.size(1))], dim=0)
        edge_x = torch.cat([node_x, node_x_other, edge_attr_padded], dim=1)

        # Classify edges into classes
        edge_scores = self.edge_classifier(edge_x)
        return edge_scores

## GAT-Training

In [ ]:
# Example usage
num_nodes = len(nodes)
num_edges = len(train_tuples)
num_node_features = 2
num_edge_features = 20
hidden_channels = 16
num_heads = 32
num_classes = 26

# Instantiate GAT classifier
model = GATClassifier(num_node_features, num_edge_features, hidden_channels, num_heads, num_classes)

# Forward pass
edge_scores_train = model(x_tensor, edge_index_tensor, edge_attr_tensor)

## GAT-Testing

In [ ]:
# Example usage
num_nodes_test = len(testnodes)
num_edges_test = len(test_tuples)

# Forward pass
edge_scores_test = model(x_test_tensor, edge_index_test_tensor, edge_attr_test_tensor)

## GAT-Author's training dataset

In [ ]:
# Example usage
num_nodes_autrain = len(autrainnodes)
num_edges_autrain = len(autrain_tuples)

num_node_features = 8
num_edge_features = 23
hidden_channels = 16
num_heads = 16
num_classes = 39

# Instantiate GAT classifier
GATmodel = GATClassifier(num_node_features, num_edge_features, hidden_channels, num_heads, num_classes)
# Forward pass
edge_scores_autrain = GATmodel(x_autrain_tensor, edge_index_autrain_tensor, edge_attr_autrain_tensor)

## GAT-Author's testing dataset

In [ ]:
# Example usage
num_nodes_autest = len(autestnodes)
num_edges_autest = len(autest_tuples)

# Forward pass
edge_scores_autest = GATmodel(x_autest_tensor, edge_index_autest_tensor, edge_attr_autest_tensor)

## Baseline Model: KNN

## KNN using User's train test

In [ ]:
import pandas as pd
# Convert edge_scores tensor to a NumPy array
edge_scores_train_array = edge_scores_train.detach().numpy()
edge_scores_test_array = edge_scores_test.detach().numpy()
# Create a DataFrame from the NumPy array
columns = [str('Dim')+str(i) for i in range(edge_scores_train_array.shape[1])]
edge_scores_train_df = pd.DataFrame(edge_scores_train_array, columns=columns)
edge_scores_test_df = pd.DataFrame(edge_scores_test_array, columns=columns)
trainLabel = train['Label']
testLabel = test['Label']

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

X_train = edge_scores_train_df
X_test = edge_scores_test_df
y_train = trainLabel
y_test = testLabel

# Initialize the KNN classifier
k = 10  # Number of neighbors
knn_classifier = KNeighborsClassifier(n_neighbors=k)

# Train the classifier
knn_classifier.fit(X_train, y_train)

# Predictions on the test set
y_pred = knn_classifier.predict(X_test)

# Calculate accuracy
report = classification_report(y_test, y_pred)
print("Classification Report: \n", report)

## Attack Sequence Trace Back

In [ ]:
#Retrieve index positions of y_pred = 1
indices_of_ones = np.where(y_pred == 1)[0]

import networkx as nx
import matplotlib.pyplot as plt
from datetime import datetime

# Create a directed graph
traceBackG = nx.DiGraph()

for i in indices_of_ones:
    src = str(testWithStimeLtime.iloc[i]['srcip']) + str(testWithStimeLtime.iloc[i]['sport'])
    dst = str(testWithStimeLtime.iloc[i]['dstip']) + str(testWithStimeLtime.iloc[i]['dsport'])
    traceBackG.add_node(src)
    traceBackG.add_node(dst)
    stime = datetime.fromtimestamp(testWithStimeLtime.iloc[i]['Stime']).strftime('%Y-%m-%d %H:%M:%S')
    ltime = datetime.fromtimestamp(testWithStimeLtime.iloc[i]['Ltime']).strftime('%Y-%m-%d %H:%M:%S')
    traceBackG.add_edge(src, dst, label="Connection Start Time: " + str(stime) + ", Connection End Time: "+ str(ltime))


# Draw the graph
pos = nx.spring_layout(traceBackG)  # Define layout
nx.draw(traceBackG, pos, with_labels=True, node_size=1000, node_color='red', font_size=12, font_weight='bold')
node_labels = nx.get_node_attributes(traceBackG, 'label')
nx.draw_networkx_labels(traceBackG, pos, labels=node_labels, font_size=12, font_weight='bold', font_color='black', 
                        bbox=dict(facecolor='white', edgecolor='none', boxstyle='round,pad=0.3'))

# Show the plot
plt.show()

## KNN Training using User prepared dataset and Testing Using Author's test dataset

In [ ]:
import pandas as pd
# Convert edge_scores tensor to a NumPy array
edge_scores_autest_array = edge_scores_autest.detach().numpy()
# Create a DataFrame from the NumPy array
#testLabeldf = pd.DataFrame(auEdgeLabel, columns=['Label'])
edge_scores_autest_df = pd.DataFrame(edge_scores_autest_array, columns=columns)
trainLabel = train['Label']
autestLabel = autest['Label']
auX_test = edge_scores_autest_df
auy_test = autestLabel 

auy_pred = knn_classifier.predict(edge_scores_autest_df)

# Calculate accuracy
report = classification_report(autestLabel, auy_pred)
print("Classification Report: \n", report)

## KNN using author's training and testing dataset

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
import pandas as pd
import numpy as np


edge_scores_autrain_array = edge_scores_autrain.detach().numpy()
edge_scores_autest_array = edge_scores_autest.detach().numpy()

columns = [str('Dim')+str(i) for i in range(edge_scores_autrain_array.shape[1])]

edge_scores_autrain_df = pd.DataFrame(edge_scores_autrain_array, columns=columns)
edge_scores_autest_df = pd.DataFrame(edge_scores_autest_array, columns=columns)


trainLabel = autrain['Label']
autestLabel = autest['Label']
auX_train = edge_scores_autrain_df
auX_test = edge_scores_autest_df

# Initialize the KNN classifier
k = 10  # Number of neighbors
knn_classifier = KNeighborsClassifier(n_neighbors=k)

# Train the classifier
knn_classifier.fit(auX_train, trainLabel)

# Predictions on the test set
y_pred = knn_classifier.predict(auX_test)

# Calculate accuracy
report = classification_report(autestLabel, y_pred)
print("Classification Report: \n", report)

## Lazy Predict Scores

## User prepared train-test

In [ ]:
# List of classifiers to include
import sklearn
import xgboost
import lightgbm
import sklearn.discriminant_analysis
from lazypredict.Supervised import LazyClassifier


classifiers = [
    ('AdaBoostClassifier', sklearn.ensemble._weight_boosting.AdaBoostClassifier),
    ('BaggingClassifier', sklearn.ensemble._bagging.BaggingClassifier),
    ('DecisionTreeClassifier', sklearn.tree._classes.DecisionTreeClassifier),
    # ('ExtraTreeClassifier', sklearn.ensemble._forest.ExtraTreeClassifier),
    # ('ExtraTreesClassifier', sklearn.ensemble._forest.ExtraTreesClassifier),
    # ('KNeighborsClassifier',  sklearn.neighbors._classification.KNeighborsClassifier),
    # ('NuSVC', sklearn.svm._classes.NuSVC),
    ('RandomForestClassifier', sklearn.ensemble._forest.RandomForestClassifier),
    # ('SVC', sklearn.svm._classes.SVC),
    ('XGBClassifier', xgboost.sklearn.XGBClassifier),
    ('LGBMClassifier', lightgbm.sklearn.LGBMClassifier)]
clf = LazyClassifier(verbose=1,ignore_warnings=True, custom_metric=None,classifiers= classifiers, predictions=True)

models,predictions = clf.fit(X_train, X_test, y_train, y_test)


In [ ]:
from sklearn.metrics import roc_auc_score
for m in predictions.columns:
    print(m)
    print(classification_report(y_test, predictions[m]))
    auc_scores = roc_auc_score(y_test, predictions[m], multi_class='ovr')  # Or multi_class='ovo' for one-vs-one
    print("AUC Scores for each class:", auc_scores)
    print('-----------------------------------------------------------------------------------')
    print('-----------------------------------------------------------------------------------')

## Lazy Predict for author's train-test dataset

In [ ]:
# List of classifiers to include
import sklearn
import xgboost
import lightgbm
import sklearn.discriminant_analysis
from lazypredict.Supervised import LazyClassifier

classifiers = [
    ('AdaBoostClassifier', sklearn.ensemble._weight_boosting.AdaBoostClassifier),
    ('BaggingClassifier', sklearn.ensemble._bagging.BaggingClassifier),
    ('DecisionTreeClassifier', sklearn.tree._classes.DecisionTreeClassifier),
    ('ExtraTreeClassifier', sklearn.ensemble._forest.ExtraTreeClassifier),
    ('ExtraTreesClassifier', sklearn.ensemble._forest.ExtraTreesClassifier),
    ('KNeighborsClassifier',  sklearn.neighbors._classification.KNeighborsClassifier),
    # ('NuSVC', sklearn.svm._classes.NuSVC),
    ('RandomForestClassifier', sklearn.ensemble._forest.RandomForestClassifier),
    # ('SVC', sklearn.svm._classes.SVC),
    ('XGBClassifier', xgboost.sklearn.XGBClassifier),
    ('LGBMClassifier', lightgbm.sklearn.LGBMClassifier)]
clf = LazyClassifier(verbose=1,ignore_warnings=True, custom_metric=None,classifiers= classifiers, predictions=True)

aumodels, aupredictions = clf.fit(auX_train, auX_test, autrainLabel, autestLabel)

In [ ]:
from sklearn.metrics import roc_auc_score
for model in aupredictions.columns:
    print(model)
    print(classification_report(autestLabel, aupredictions[model]))
    auc_scores = roc_auc_score(autestLabel, aupredictions[model], multi_class='ovr')  # Or multi_class='ovo' for one-vs-one
    print("AUC Scores for each class:", auc_scores)
    print('-----------------------------------------------------------------------------------')
    print('-----------------------------------------------------------------------------------')

In [ ]:
aumodels

# GCN

## GCN Model Definition

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

class GCNClassifier(torch.nn.Module):
    def __init__(self, num_node_features, num_edge_features, hidden_channels, num_classes):
        super(GCNClassifier, self).__init__()
        self.conv1 = GCNConv(num_node_features, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        # self.edge_classifier = nn.Linear(hidden_channels * 2 + num_edge_features, num_classes)
        self.edge_classifier = nn.Sequential(
        nn.Linear(hidden_channels * 2 + num_edge_features, hidden_channels),  # Linear layer
        nn.ReLU(),  # ReLU activation function
        nn.Linear(hidden_channels, num_classes)  # Linear layer
        )


    def forward(self, x, edge_index, edge_attr):
        # Apply graph convolutional layers
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        x = F.relu(x)

        # Concatenate node features with edge features
        num_edges = edge_index.size(1)
        node_x = x[edge_index[0]]
        node_x_other = x[edge_index[1]]
        edge_attr_padded = torch.cat([edge_attr, torch.zeros(num_edges - edge_attr.size(0), edge_attr.size(1))], dim=0)
        edge_x = torch.cat([node_x, node_x_other, edge_attr_padded], dim=1)

        # Classify edges into classes
        edge_scores = self.edge_classifier(edge_x)
        return edge_scores

## GCN-Training using user prepared dataset

In [ ]:
num_nodes = len(nodes)
num_edges = len(train_tuples)
num_node_features = 2
num_edge_features = 20
hidden_channels = 8
num_classes = 26

GCNmodel1 = GCNClassifier(num_node_features, num_edge_features, hidden_channels, num_classes)


GCNedge_scores_train = GCNmodel1(x_tensor, edge_index_tensor, edge_attr_tensor)

## GCN-Testing with user prepared dataset

In [ ]:
num_nodes_test = len(testnodes)
num_edges_test = len(test_tuples)

# Forward pass
GCNedge_scores_test = GCNmodel1(x_test_tensor, edge_index_test_tensor, edge_attr_test_tensor)

## GCN-Training using Author's Train set

In [ ]:
num_nodes_autrain = len(autrainnodes)
num_edges_autrain = len(autrain_tuples)
num_node_features = 8
num_edge_features = 23
hidden_channels = 8
num_classes = 64

GCNmodel2 = GCNClassifier(num_node_features, num_edge_features, hidden_channels, num_classes)
GCNedge_scores_autrain = GCNmodel2(x_autrain_tensor, edge_index_autrain_tensor, edge_attr_autrain_tensor)

## GCN-Testing using Author's Test set

In [ ]:
# Example usage
num_nodes_autest = len(autestnodes)
num_edges_autest = len(autest_tuples)


# Forward pass
GCNedge_scores_autest = GCNmodel2(x_autest_tensor, edge_index_autest_tensor, edge_attr_autest_tensor)

## Baseline Model: KNN

## KNN using User's train-test

In [ ]:
import pandas as pd
# Convert edge_scores tensor to a NumPy array
GCNedge_scores_train_array = GCNedge_scores_train.detach().numpy()
GCNedge_scores_test_array = GCNedge_scores_test.detach().numpy()
# Create a DataFrame from the NumPy array
columns = [str('Dim')+str(i) for i in range(GCNedge_scores_train_array.shape[1])]
GCNedge_scores_train_df = pd.DataFrame(GCNedge_scores_train_array, columns=columns)
GCNedge_scores_test_df = pd.DataFrame(GCNedge_scores_test_array, columns=columns)
trainLabel = train['Label']
testLabel = test['Label']

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

GCNX_train = GCNedge_scores_train_df
GCNX_test = GCNedge_scores_test_df
y_train = trainLabel
y_test = testLabel

# Initialize the KNN classifier
k = 10  # Number of neighbors
knn_classifier = KNeighborsClassifier(n_neighbors=k)

# Train the classifier
knn_classifier.fit(GCNX_train, y_train)

# Predictions on the test set
y_pred = knn_classifier.predict(GCNX_test)

# Calculate accuracy
report = classification_report(y_test, y_pred)
print("Classification Report: \n", report)

## KNN training using user's dataset and testing using author's test dataset

In [ ]:
import pandas as pd
# Convert edge_scores tensor to a NumPy array
GCNedge_scores_autest_array = GCNedge_scores_autest.detach().numpy()
# Create a DataFrame from the NumPy array
#testLabeldf = pd.DataFrame(auEdgeLabel, columns=['Label'])
GCNedge_scores_autest_df = pd.DataFrame(GCNedge_scores_autest_array, columns=columns)
trainLabel = train['Label']
autestLabel = autest['Label']
auX_test = GCNedge_scores_autest_df
auy_test = autestLabel 

auy_pred = knn_classifier.predict(GCNedge_scores_autest_df)

# Calculate accuracy
report = classification_report(autestLabel, auy_pred)
print("Classification Report: \n", report)

## KNN using Author's train-test Set

In [ ]:
GCNedge_scores_autrain_array = GCNedge_scores_autrain.detach().numpy()
GCNedge_scores_autest_array = GCNedge_scores_autest.detach().numpy()

columns = [str('Dim')+str(i) for i in range(GCNedge_scores_autrain_array.shape[1])]

GCNedge_scores_autrain_df = pd.DataFrame(GCNedge_scores_autrain_array, columns=columns)
GCNedge_scores_autest_df = pd.DataFrame(GCNedge_scores_autest_array, columns=columns)


trainLabel = autrain['Label']
autestLabel = autest['Label']
auX_train = GCNedge_scores_autrain_df
auX_test = GCNedge_scores_autest_df

# Initialize the KNN classifier
k = 10  # Number of neighbors
knn_classifier = KNeighborsClassifier(n_neighbors=k)

# Train the classifier
knn_classifier.fit(auX_train, trainLabel)
# # Get training score
# training_score = knn_classifier.score(auX_train, trainLabel)
# print("Training score:", training_score)
# # Predictions on the test set
# y_predtrain = knn_classifier.predict(auX_train)

# # Calculate accuracy
# report = classification_report(trainLabel, y_predtrain)
# print("Classification Report for Training: \n", report)

# Predictions on the test set
y_pred = knn_classifier.predict(auX_test)

# Calculate accuracy
report = classification_report(autestLabel, y_pred)
print("Classification Report: \n", report)

## Lazy Predict

In [ ]:
# List of classifiers to include
import sklearn
import xgboost
import lightgbm
import sklearn.discriminant_analysis
from lazypredict.Supervised import LazyClassifier

classifiers = [
    ('AdaBoostClassifier', sklearn.ensemble._weight_boosting.AdaBoostClassifier),
    ('BaggingClassifier', sklearn.ensemble._bagging.BaggingClassifier),
    ('DecisionTreeClassifier', sklearn.tree._classes.DecisionTreeClassifier),
    ('ExtraTreeClassifier', sklearn.ensemble._forest.ExtraTreeClassifier),
    ('ExtraTreesClassifier', sklearn.ensemble._forest.ExtraTreesClassifier),
    ('KNeighborsClassifier',  sklearn.neighbors._classification.KNeighborsClassifier),
    # ('NuSVC', sklearn.svm._classes.NuSVC),
    ('RandomForestClassifier', sklearn.ensemble._forest.RandomForestClassifier),
    # ('SVC', sklearn.svm._classes.SVC),
    ('XGBClassifier', xgboost.sklearn.XGBClassifier),
    ('LGBMClassifier', lightgbm.sklearn.LGBMClassifier)]
clf = LazyClassifier(verbose=1,ignore_warnings=True, custom_metric=None,classifiers= classifiers, predictions=True)

aumodels, aupredictions = clf.fit(auX_train, auX_test, autrainLabel, autestLabel)

In [ ]:
from sklearn.metrics import roc_auc_score
for model in aupredictions.columns:
    print(model)
    print(classification_report(autestLabel, aupredictions[model]))
    auc_scores = roc_auc_score(autestLabel, aupredictions[model], multi_class='ovr')  # Or multi_class='ovo' for one-vs-one
    print("AUC Scores for each class:", auc_scores)
    print('-----------------------------------------------------------------------------------')
    print('-----------------------------------------------------------------------------------')

In [ ]:
aumodels

# GraphSAGE

## GraphSAGE Model Declaration

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
class GraphSAGEClassifier(torch.nn.Module):
    def __init__(self, num_node_features, num_edge_features, hidden_channels, num_classes):
        super(GraphSAGEClassifier, self).__init__()
        self.conv1 = SAGEConv(num_node_features, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.edge_classifier = nn.Linear(hidden_channels * 2 + num_edge_features, num_classes)

    def forward(self, x, edge_index, edge_attr):
        # Apply GraphSAGE convolutional layers
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        x = F.relu(x)

        # Concatenate node features with edge features
        num_edges = edge_index.size(1)
        node_x = x[edge_index[0]]
        node_x_other = x[edge_index[1]]
        edge_attr_padded = torch.cat([edge_attr, torch.zeros(num_edges - edge_attr.size(0), edge_attr.size(1))], dim=0)
        edge_x = torch.cat([node_x, node_x_other, edge_attr_padded], dim=1)

        # Classify edges into binary class using BCELogit
        edge_scores = self.edge_classifier(edge_x)
        return edge_scores

## GraphSAGE- User's train-test

In [ ]:
num_nodes = len(nodes)
num_edges = len(train_tuples)
num_node_features = 2
num_edge_features = 20
hidden_channels = 8
num_classes = 26

GraphSAGEmodel1 = GraphSAGEClassifier(num_node_features, num_edge_features, hidden_channels, num_classes)


GraphSAGEedge_scores_train = GraphSAGEmodel1(x_tensor, edge_index_tensor, edge_attr_tensor)

In [ ]:
num_nodes_test = len(testnodes)
num_edges_test = len(test_tuples)

# Forward pass
GraphSAGEedge_scores_test = GraphSAGEmodel1(x_test_tensor, edge_index_test_tensor, edge_attr_test_tensor)

## GraphSAGE-Author's train-test

In [ ]:
num_nodes_autrain = len(autrainnodes)
num_edges_autrain = len(autrain_tuples)
num_node_features = 8
num_edge_features = 23
hidden_channels = 8
num_classes = 39

GraphSAGEmodel2 = GraphSAGEClassifier(num_node_features, num_edge_features, hidden_channels, num_classes)
GraphSAGEedge_scores_autrain = GraphSAGEmodel2(x_autrain_tensor, edge_index_autrain_tensor, edge_attr_autrain_tensor)

In [ ]:
num_nodes_autest = len(autestnodes)
num_edges_autest = len(autest_tuples)


# Forward pass
GraphSAGEedge_scores_autest = GraphSAGEmodel2(x_autest_tensor, edge_index_autest_tensor, edge_attr_autest_tensor)

## Baseline Model: KNN

## KNN using User's train-test

In [ ]:
import pandas as pd
# Convert edge_scores tensor to a NumPy array
GraphSAGEedge_scores_train_array = GraphSAGEedge_scores_train.detach().numpy()
GraphSAGEedge_scores_test_array = GraphSAGEedge_scores_test.detach().numpy()
# Create a DataFrame from the NumPy array
columns = [str('Dim')+str(i) for i in range(GraphSAGEedge_scores_train_array.shape[1])]
GraphSAGEedge_scores_train_df = pd.DataFrame(GraphSAGEedge_scores_train_array, columns=columns)
GraphSAGEedge_scores_test_df = pd.DataFrame(GraphSAGEedge_scores_test_array, columns=columns)
trainLabel = train['Label']
testLabel = test['Label']

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

GraphSAGEX_train = GraphSAGEedge_scores_train_df
GraphSAGEX_test = GraphSAGEedge_scores_test_df
y_train = trainLabel
y_test = testLabel

# Initialize the KNN classifier
k = 10  # Number of neighbors
knn_classifier = KNeighborsClassifier(n_neighbors=k)

# Train the classifier
knn_classifier.fit(GraphSAGEX_train, y_train)

# Predictions on the test set
y_pred = knn_classifier.predict(GraphSAGEX_test)

# Calculate accuracy
report = classification_report(y_test, y_pred)
print("Classification Report: \n", report)

## Training using user's dataset but testing using author's test set

In [ ]:
import pandas as pd
# Convert edge_scores tensor to a NumPy array
GraphSAGEedge_scores_autest_array = GraphSAGEedge_scores_autest.detach().numpy()
# Create a DataFrame from the NumPy array
#testLabeldf = pd.DataFrame(auEdgeLabel, columns=['Label'])
GraphSAGEedge_scores_autest_df = pd.DataFrame(GraphSAGEedge_scores_autest_array, columns=columns)
trainLabel = train['Label']
autestLabel = autest['Label']
auX_test = GraphSAGEedge_scores_autest_df
auy_test = autestLabel 

auy_pred = knn_classifier.predict(GraphSAGEedge_scores_autest_df)

# Calculate accuracy
report = classification_report(autestLabel, auy_pred)
print("Classification Report: \n", report)

## KNN Author's train-test

In [ ]:
GraphSAGEedge_scores_autrain_array = GraphSAGEedge_scores_autrain.detach().numpy()
GraphSAGEedge_scores_autest_array = GraphSAGEedge_scores_autest.detach().numpy()

GraphSAGEedge_scores_autrain_df = pd.DataFrame(GraphSAGEedge_scores_autrain_array, columns=columns)
GraphSAGEedge_scores_autest_df = pd.DataFrame(GraphSAGEedge_scores_autest_array, columns=columns)


trainLabel = autrain['Label']
autestLabel = autest['Label']
auX_train = GraphSAGEedge_scores_autrain_df
auX_test = GraphSAGEedge_scores_autest_df

# Initialize the KNN classifier
k = 15  # Number of neighbors
knn_classifier = KNeighborsClassifier(n_neighbors=k)

# Train the classifier
knn_classifier.fit(auX_train, trainLabel)

# Predictions on the test set
y_pred = knn_classifier.predict(auX_test)

# Calculate accuracy
report = classification_report(autestLabel, y_pred)
print("Classification Report: \n", report)

## Lazy Predict

In [ ]:
# List of classifiers to include
import sklearn
import xgboost
import lightgbm
import sklearn.discriminant_analysis
from lazypredict.Supervised import LazyClassifier

classifiers = [
    ('AdaBoostClassifier', sklearn.ensemble._weight_boosting.AdaBoostClassifier),
    ('BaggingClassifier', sklearn.ensemble._bagging.BaggingClassifier),
    ('DecisionTreeClassifier', sklearn.tree._classes.DecisionTreeClassifier),
    ('ExtraTreeClassifier', sklearn.ensemble._forest.ExtraTreeClassifier),
    ('ExtraTreesClassifier', sklearn.ensemble._forest.ExtraTreesClassifier),
    ('KNeighborsClassifier',  sklearn.neighbors._classification.KNeighborsClassifier),
    # ('NuSVC', sklearn.svm._classes.NuSVC),
    ('RandomForestClassifier', sklearn.ensemble._forest.RandomForestClassifier),
    # ('SVC', sklearn.svm._classes.SVC),
    ('XGBClassifier', xgboost.sklearn.XGBClassifier),
    ('LGBMClassifier', lightgbm.sklearn.LGBMClassifier)]
clf = LazyClassifier(verbose=1,ignore_warnings=True, custom_metric=None,classifiers= classifiers, predictions=True)

aumodels, aupredictions = clf.fit(auX_train, auX_test, autrainLabel, autestLabel)

In [ ]:
from sklearn.metrics import roc_auc_score
for model in aupredictions.columns:
    print(model)
    print(classification_report(autestLabel, aupredictions[model]))
    auc_scores = roc_auc_score(autestLabel, aupredictions[model], multi_class='ovr')  # Or multi_class='ovo' for one-vs-one
    print("AUC Scores for each class:", auc_scores)
    print('-----------------------------------------------------------------------------------')
    print('-----------------------------------------------------------------------------------')

In [ ]:
aumodels